# RSNA Knee Abnormality Detection — Phase 2: LLM Report Labeler
### Zero-Shot Multilingual NLI (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`)

This notebook extracts high-fidelity soft probability labels from 4,407 multilingual radiology reports:
1. **Positive Hypothesis:** `The MRI shows [finding description].` $\to p_{\text{pos}}$
2. **Negation Hypothesis:** `There is no evidence of [finding description].` $\to p_{\text{neg}}$
3. **Soft Target:** $\text{clip}(p_{\text{pos}} - 0.5 \times p_{\text{neg}}, 0.05, 0.95)$
4. **Evaluation:** Compares LLM macro ROC-AUC against baseline regex labeler on the 58 gold studies.

In [ ]:
import os
import sys
import time
import glob
import re
import unicodedata
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

# Optional offline wheel installation if wheels dataset is attached
wheels = glob.glob('/kaggle/input/**/transformers*.whl', recursive=True)
if wheels:
    print(f'[+] Installing offline wheel: {wheels[0]}')
    os.system(f'pip install {wheels[0]} --no-index --find-links=/kaggle/input')

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] Using device: {device}')


In [ ]:
FINDINGS = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's Cyst",
    'Contusion',
    'Fracture',
]

FINDING_DESCRIPTIONS = {
    'ACL': 'anterior cruciate ligament tear or abnormality',
    'MCL': 'medial collateral ligament tear or abnormality',
    'Medial Meniscus': 'medial meniscus tear or abnormality',
    'Lateral Meniscus': 'lateral meniscus tear or abnormality',
    'Medial OA': 'medial tibiofemoral osteoarthritis or joint space narrowing',
    'Lateral OA': 'lateral tibiofemoral osteoarthritis or joint space narrowing',
    'PF OA': 'patellofemoral osteoarthritis or cartilage loss',
    'Effusion': 'knee joint effusion or fluid',
    'Synovitis': 'synovitis or synovial thickening',
    "Baker's Cyst": 'Baker cyst or popliteal cyst',
    'Contusion': 'bone contusion or bone marrow edema',
    'Fracture': 'fracture',
}

# Locate mDeBERTa model weights offline
model_cands = [
    '/kaggle/input/mdeberta-weights',
    '/kaggle/input/mdeberta-v3-base-mnli-xnli',
    'weights/mdeberta',
    'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli',
]

model_path = next((p for p in model_cands if os.path.exists(p)), model_cands[0])
print(f'[+] Loading mDeBERTa from: {model_path}')

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

# Determine entailment index from label mapping
label2id = getattr(model.config, 'label2id', {})
entail_idx = 0
for k, v in label2id.items():
    if 'entail' in k.lower():
        entail_idx = v
        break
print(f'[+] Entailment class index: {entail_idx}')


In [ ]:
data_root = '/kaggle/input/rsna-knee-abnormality-detection' if os.path.exists('/kaggle/input/rsna-knee-abnormality-detection') else 'data'
train_csv = os.path.join(data_root, 'train.csv')
df_train = pd.read_csv(train_csv)
print(f'[+] Loaded {len(df_train)} reports from {train_csv}')

def normalize_text(text: str) -> str:
    if not isinstance(text, str): return ''
    nfkd = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in nfkd if not unicodedata.combining(c)).replace('ß', 'ss').lower()

# Extract NLI labels in batches
BATCH_SIZE = 32
all_results = []

for idx in tqdm(range(len(df_train)), desc='Labeling reports with mDeBERTa'):
    row = df_train.iloc[idx]
    study_uid = row['StudyInstanceUID']
    text = normalize_text(str(row.get('report', '')))
    
    if not text.strip():
        row_out = {'StudyInstanceUID': study_uid}
        for f in FINDINGS:
            row_out[f'{f}_target'] = 0.15
            row_out[f'{f}_weight'] = 0.30
            row_out[f] = 0.15
        all_results.append(row_out)
        continue
    
    # Construct pairs for all 12 findings (24 pairs per study)
    pairs = []
    for f in FINDINGS:
        desc = FINDING_DESCRIPTIONS[f]
        pairs.append((text, f'The MRI shows {desc}.'))
        pairs.append((text, f'There is no evidence of {desc}.'))
    
    # Batch forward pass
    premises = [p[0] for p in pairs]
    hypotheses = [p[1] for p in pairs]
    inputs = tokenizer(premises, hypotheses, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[:, entail_idx].cpu().numpy()
    
    row_out = {'StudyInstanceUID': study_uid}
    for f_idx, f in enumerate(FINDINGS):
        p_pos = float(probs[2 * f_idx])
        p_neg = float(probs[2 * f_idx + 1])
        soft_target = float(np.clip(p_pos - 0.5 * p_neg, 0.05, 0.95))
        weight = float(np.clip(max(p_pos, p_neg) * 1.2, 0.30, 1.00))
        row_out[f'{f}_target'] = soft_target
        row_out[f'{f}_weight'] = weight
        row_out[f'{f}_p_pos'] = p_pos
        row_out[f'{f}_p_neg'] = p_neg
        row_out[f] = soft_target
    
    all_results.append(row_out)

df_llm = pd.DataFrame(all_results)
out_path = '/kaggle/working/weak_labels_llm.csv'
df_llm.to_csv(out_path, index=False)
print(f'[OK] Saved LLM weak labels to {out_path} with shape {df_llm.shape}')


In [ ]:
# Evaluate LLM vs Regex on 58 Gold Studies
gold_mask = df_train[FINDINGS].notna().any(axis=1)
df_gold = df_train[gold_mask].copy()
print(f'[+] Evaluating on {len(df_gold)} gold-label studies.')

# Load regex labels if available
regex_cands = ['/kaggle/input/weak-labels-v0/weak_labels_v0.csv', 'results/weak_labels_v0.csv']
regex_path = next((p for p in regex_cands if os.path.exists(p)), None)

if regex_path:
    df_reg = pd.read_csv(regex_path)
    m_reg = pd.merge(df_gold, df_reg, on='StudyInstanceUID', suffixes=('_gold', '_reg'))
    m_llm = pd.merge(df_gold, df_llm, on='StudyInstanceUID', suffixes=('_gold', '_llm'))
    
    print('\n' + '=' * 65)
    print(f'{"Finding":20s} {"Regex AUC":>12s} {"LLM AUC":>12s} {"Delta":>12s}')
    print('=' * 65)
    
    r_aucs, l_aucs = [], []
    for f in FINDINGS:
        gt = m_reg[f'{f}_gold'].values
        if len(np.unique(gt)) < 2: continue
        ra = roc_auc_score(gt, m_reg[f'{f}_target'].values)
        la = roc_auc_score(gt, m_llm[f'{f}_target'].values)
        r_aucs.append(ra)
        l_aucs.append(la)
        print(f'{f:20s} {ra:>12.4f} {la:>12.4f} {(la - ra):>+12.4f}')
    
    macro_r = np.mean(r_aucs)
    macro_l = np.mean(l_aucs)
    delta = macro_l - macro_r
    print('-' * 65)
    print(f'{"MACRO ROC-AUC":20s} {macro_r:>12.4f} {macro_l:>12.4f} {delta:>+12.4f}')
    print('=' * 65)
    
    if delta >= 0.03:
        print(f'[OK] SUCCESS: LLM beat Regex by {delta:+.4f} (>= +0.03 target)! Using LLM labels.')
    else:
        print(f'[!] LLM delta ({delta:+.4f}) < +0.03. Recommended: Ensemble (0.5 * regex + 0.5 * llm).')
else:
    print('[!] Regex baseline CSV not found for comparison. Proceeding with LLM weak labels.')
